# Day 10 — ILT 2: Introduction to Orchestration — Need, DAG Concepts and Workflow Design

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Time** | 3:00 PM – 4:00 PM |
| **Builds on** | Every ingestion, Silver, and Gold notebook from Day 2–9, plus this morning's Day 10 ILT 1 (SCD Type 1/2 MERGE) |
| **Real tables referenced** | `gbmart.bronze.*`, `gbmart.silver.*`, `gbmart.gold.*` — the full medallion pipeline built so far |
| **Duration** | 60 minutes |
| **Mode** | Instructor-led — concept walkthrough + a real, runnable DAG/topological-sort demo in pure Python. No learner hands-on redo today; Day 11's Hands-On is where you build the real Databricks Workflow. |

### Learning Objectives

By the end of this session, students will be able to:

1. Explain *why* orchestration becomes necessary once a pipeline has more than a couple of notebooks with dependencies between them
2. Define a DAG (Directed Acyclic Graph) in terms of **nodes** (tasks) and **edges** (dependencies), and explain why cycles make a graph unschedulable
3. Identify **fan-out** (independent tasks that can run in parallel) and **fan-in** (a task that must wait on multiple upstream tasks) in GlobalMart's real pipeline
4. Explain how a Databricks Job/Workflow represents a DAG — a job as a list of tasks, each with a `task_key`, a `notebook_task`/`pipeline_task`, and a `depends_on` list
5. Run a real topological sort over GlobalMart's actual task graph, in pure Python, and explain why a cyclic graph raises an error instead of producing a schedule

---

**INSTRUCTOR NOTE:** This is a concept session with real, executing code — not a slides-only lecture, and not a hands-on lab either. Every code cell below is plain Python (no Spark session, no Databricks SDK, no network call). The goal is for students to see the actual mechanics of a DAG and a topological sort with their own eyes, on GlobalMart's real task names, before Day 11 has them click through the same structure in the Databricks Workflows UI.

**Safety, stated up front:** nothing in this notebook creates, starts, or triggers a Databricks Job, Workflow, Pipeline, SQL Warehouse, or cluster. There is no Jobs/Pipelines REST API call, no job-creation SDK call, and no `dbutils`-based job control anywhere in this file — every cell is safe to run, and safe to re-run, on any machine with Python, whether or not it is even attached to Databricks.

---
## Section 1: Why Orchestration Is Needed

**INSTRUCTOR NOTE:** Before showing any code, ask the class: *"Since Day 2, how many notebooks have you personally had to run, in the correct order, for a trustworthy number to land in `fact_sales`?"* Most won't have counted. Count it together, live — the table below is the answer, and it's the entire motivation for this session.

### The pipeline you've actually built, Day 2 – Day 10

| Layer | Real notebooks you've already run | Lands in |
|---|---|---|
| Ingestion — Postgres CDC | `Day2_HOL1_Lakeflow_Connect_Storage_Credentials`, `Day2_HOL2_CDC_PostgreSQL_WAL` | `gbmart.bronze.orders`, `order_items` |
| Ingestion — ADLS Autoloader | `Day3_HOL2_ADLS_AutoLoader_Bronze_Customers_Payments`, `Day4_HOL1_Build_Bronze_Layer_All_Sources` | `gbmart.bronze.customers`, `products`, `addresses`, `payments`, `payment_methods` |
| Silver | `Day5_HOL1_Transformations_DQ_Quarantine`, `Day5_HOL2_Build_Silver_Layer_All_Sources` | All 7 `gbmart.silver.*` tables |
| Gold | `Day7_HOL1_Build_Dimensions_Bridge_Table`, `Day7_HOL2_Build_Gold_Layer_Fact_Sales` | 6 dimensions + `gbmart.gold.fact_sales` |
| Incremental / SCD | `Day9_HOL1_Watermark_Incremental_Load_Control_Table`, `Day9_HOL2_Handling_Incremental_Data_Bronze_Silver`, `Day10_HOL1_SCD_Type2_Dim_Customer_MERGE`, `Day10_HOL2_Incremental_Gold_Refresh_Fact_MERGE` | Keeps Silver/Gold current without full reprocessing |

That's **12 real hands-on notebooks** that build or refresh a real `gbmart` table — before counting a single ILT demo, and all of it already behind you by this point on Day 10.

> **Callback — it's 2 real pathways, not 4.** Day 3 ILT 1 and Day 4 ILT 1 already established this, explicitly: GlobalMart's production pipeline has exactly **two** real ingestion sources — Postgres CDC via Lakeflow Connect, and ADLS Autoloader. The REST API and Neo4j/GraphDB patterns from Day 3 were deliberate **side-explorations** — useful patterns for your general toolkit, landing in a `sandbox/` path, never touching a `gbmart.bronze` table. That distinction matters here more than anywhere else so far: **an orchestration job only schedules what's actually in production.** If a notebook doesn't write to a real `gbmart` table, it has no business having a `task_key` in this DAG — which is exactly the design you'll see in Section 4.

### The manual pain, made concrete

Imagine running this by hand, every single morning, forever:

```
1. Confirm the Lakeflow Connect CDC pipeline succeeded for orders + order_items
2. Kick off the Autoloader ingestion for customers, products, addresses, payments, payment_methods
3. Wait for every one of the above to finish
4. Run all 7 Silver notebooks -- but only once each one's own Bronze table has landed
5. Wait for all 7 Silver tables to finish
6. Run all 6 dimension builds, then fact_sales last
7. Hope nobody fat-fingered the order, and hope nothing upstream silently failed
```

Nothing above is conceptually hard. It's just **a lot of ordering rules for a human to hold in their head and babysit, every day, forever.** That is the exact gap orchestration closes: a scheduler that already knows the dependency rules, runs everything that's ready at the same time, waits for the right things, and fails loudly instead of silently — instead of a person watching a clock.

---
## Section 2: DAG Concepts — Nodes, Edges, Fan-Out, and Fan-In

A **DAG (Directed Acyclic Graph)** is the data structure every real orchestrator — Databricks Workflows, Airflow, Dagster — uses under the hood to represent a pipeline.

| Term | Meaning | GlobalMart example |
|---|---|---|
| **Node** | One unit of work — a task | "Ingest `gbmart.bronze.orders`" |
| **Edge** | A directed dependency: "B cannot start until A finishes" | `bronze_orders → silver_orders` |
| **Directed** | Edges have a direction — order matters | Never the reverse: Silver can't run before Bronze |
| **Acyclic** | No path ever loops back to where it started | See below |

### Why cycles are invalid

If `task_a` depends on `task_b`, and `task_b` depends on `task_a`, **neither can ever start** — each is waiting on the other to go first. This isn't a slow-performance problem, it's a logical impossibility: there is no valid task to run first. A scheduler must be able to *prove* a graph has no cycle before it can compute any run order at all — that proof is exactly what topological sort does (Section 5), and exactly why it must raise an error instead of hanging forever when a cycle exists (Section 6).

### Fan-out: independent work, free to run in parallel

**Fan-out** is any point in the DAG where multiple tasks become runnable at the same time because none of them depends on any of the others.

> GlobalMart's real fan-out: the CDC-driven tasks (`bronze_orders`, `bronze_order_items`) and the five Autoloader-driven tasks (`bronze_customers`, `bronze_products`, `bronze_addresses`, `bronze_payments`, `bronze_payment_methods`) share **zero dependencies on each other**. All seven can start at the same moment, on the same cluster or on seven different ones, with no coordination required between them.

### Fan-in: a task that has to wait

**Fan-in**, loosely, is any point where a task cannot start until an upstream task finishes — even a single dependency is a small fan-in gate in that sense: `silver_orders` cannot start one second before `bronze_orders` finishes, no matter how idle the cluster is. The stricter, more consequential case is a task with **multiple** upstream dependencies at once. GlobalMart's clearest example: `gold_fact_sales` has to wait on **six** upstream tasks simultaneously — five Silver tables (`order_items`, `orders`, `products`, `address`, `payments`) plus `gold_dim_date`. Miss, delay, or fail any one of those six, and `fact_sales` cannot start.

> **A design choice, not a law.** This session gives every dimension a narrow, single-parent dependency — its own Silver table, and nothing else. Plenty of real orchestration jobs instead make *every* Gold task depend on the entire Silver layer finishing first — coarser, easier to reason about, and safer against half-finished Silver runs, at the cost of avoidable waiting (`dim_payment_method` would sit idle waiting on `silver_orders`, even though it never reads that table). `fact_sales` already shows the narrower, more surgical style for real; Discussion Question 2 at the end asks you to work out what changes if every Gold task took the coarser approach instead.

Fan-out and fan-in aren't exotic patterns reserved for huge companies — they're the normal shape of almost every real data pipeline with more than one source. Section 4 builds GlobalMart's actual graph in code; Section 5 lets a real algorithm compute a valid order automatically, instead of a person reasoning through it by hand every morning.

---
## Section 3: Workflow Design — How Databricks Jobs Model a DAG

A **Databricks Job** (the UI now calls this surface "Workflows") is, underneath the drag-and-drop screen, just **a JSON document describing a DAG**: a name, a list of tasks, and the compute those tasks run on.

### The fields that matter today

| Field | What it holds | GlobalMart example |
|---|---|---|
| `task_key` | A short, unique name for this task within the job | `"silver_orders"` |
| `depends_on` | Which upstream `task_key`(s) must succeed first | `["bronze_orders"]` |
| `notebook_task.notebook_path` | The workspace path of the notebook this task runs | `/Repos/globalmart-prod/silver/build_silver_orders` |
| `pipeline_task.pipeline_id` | Used instead of `notebook_task` when the task is really a Lakeflow/DLT pipeline run, not a notebook | GlobalMart's real `bronze_orders` task — it's driven by the `orders_data_ingestion_cdc` pipeline, not a plain notebook |

A task can also be a `sql_task`, a `dbt_task`, a `python_wheel_task`, and others — but the **shape of the DAG itself never changes**: every task type still carries a `task_key` and a `depends_on` list. That's the part this session focuses on.

### The whole job, reduced to its essence

```
job = {
    "name": "...",
    "tasks": [ {task}, {task}, {task}, ... ],   # every node in the DAG
    "job_clusters": [ ... ],                     # shared compute definitions
    "schedule": { "quartz_cron_expression": "...", "timezone_id": "..." }
}
```

Section 4 builds exactly this shape — as a plain Python list of dicts, for GlobalMart's real pipeline. Section 7 converts it into the literal nested JSON a real job-creation call would expect, purely to look at — never to send.

---
## Section 4: Building GlobalMart's Real Task Graph

The two cells below are **plain Python — no Spark session, no Databricks SDK import, no network call.** Nothing in this section can cost anything to run, on any machine.

The graph: **2 ingestion pathways → 7 Bronze tables → 7 Silver tables → 6 dimensions + `fact_sales`.** Every `target_table` below is a real table name from this course's `gbmart` catalog (Day 2–7); every `depends_on` edge mirrors a real dependency you already built by hand.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GlobalMart's real task graph: 2 ingestion pathways -> 7 Bronze tables ->
# 7 Silver tables -> 6 dimensions + fact_sales. Plain Python list of dicts --
# no Spark session, no Databricks SDK, no network call. This cell cannot
# cost anything to run.
#
# Shape matches the real Databricks Jobs API's task schema: task_key,
# depends_on, and either notebook_path (Autoloader/Silver/Gold notebooks)
# or pipeline_id (the 2 tasks really driven by Lakeflow Connect).
# ═══════════════════════════════════════════════════════════════════════════

globalmart_tasks = [

    # -- Bronze: Postgres (Supabase) via Lakeflow Connect CDC --------------
    # Real pipeline name from Day 2: orders_data_ingestion_cdc.
    # In a real job this is a pipeline_task, not a notebook_task.
    {
        "task_key": "bronze_orders",
        "layer": "bronze",
        "depends_on": [],
        "pipeline_id": "orders_data_ingestion_cdc",
        "target_table": "gbmart.bronze.orders",
    },
    {
        "task_key": "bronze_order_items",
        "layer": "bronze",
        "depends_on": [],
        "pipeline_id": "orders_data_ingestion_cdc",
        "target_table": "gbmart.bronze.order_items",
    },

    # -- Bronze: ADLS Autoloader (5 entities, Day 3-5) ----------------------
    {
        "task_key": "bronze_customers",
        "layer": "bronze",
        "depends_on": [],
        "notebook_path": "/Repos/globalmart-prod/bronze/ingest_customers",
        "target_table": "gbmart.bronze.customers",
    },
    {
        "task_key": "bronze_products",
        "layer": "bronze",
        "depends_on": [],
        "notebook_path": "/Repos/globalmart-prod/bronze/ingest_products",
        "target_table": "gbmart.bronze.products",
    },
    {
        "task_key": "bronze_addresses",
        "layer": "bronze",
        "depends_on": [],
        "notebook_path": "/Repos/globalmart-prod/bronze/ingest_addresses",
        "target_table": "gbmart.bronze.addresses",
    },
    {
        "task_key": "bronze_payments",
        "layer": "bronze",
        "depends_on": [],
        "notebook_path": "/Repos/globalmart-prod/bronze/ingest_payments",
        "target_table": "gbmart.bronze.payments",
    },
    {
        "task_key": "bronze_payment_methods",
        "layer": "bronze",
        "depends_on": [],
        "notebook_path": "/Repos/globalmart-prod/bronze/ingest_payment_methods",
        "target_table": "gbmart.bronze.payment_methods",
    },

    # -- Silver: each waits ONLY on its own Bronze table (Day 5) -----------
    {
        "task_key": "silver_orders",
        "layer": "silver",
        "depends_on": ["bronze_orders"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_orders",
        "target_table": "gbmart.silver.orders",
    },
    {
        "task_key": "silver_order_items",
        "layer": "silver",
        "depends_on": ["bronze_order_items"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_order_items",
        "target_table": "gbmart.silver.order_items",
    },
    {
        "task_key": "silver_customers",
        "layer": "silver",
        "depends_on": ["bronze_customers"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_customers",
        "target_table": "gbmart.silver.customers",
    },
    {
        "task_key": "silver_products",
        "layer": "silver",
        "depends_on": ["bronze_products"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_products",
        "target_table": "gbmart.silver.products",
    },
    {
        "task_key": "silver_address",
        "layer": "silver",
        "depends_on": ["bronze_addresses"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_address",
        "target_table": "gbmart.silver.address",   # Silver renames addresses -> address, singular
    },
    {
        "task_key": "silver_payments",
        "layer": "silver",
        "depends_on": ["bronze_payments"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_payments",
        "target_table": "gbmart.silver.payments",
    },
    {
        "task_key": "silver_payment_methods",
        "layer": "silver",
        "depends_on": ["bronze_payment_methods"],
        "notebook_path": "/Repos/globalmart-prod/silver/build_silver_payment_methods",
        "target_table": "gbmart.silver.payment_methods",
    },

    # -- Gold: 6 dimensions, each from its one real Silver source (Day 6-7) -
    {
        "task_key": "gold_dim_customer",
        "layer": "gold",
        "depends_on": ["silver_customers"],
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_customer",
        "target_table": "gbmart.gold.dim_customer",
    },
    {
        "task_key": "gold_dim_product",
        "layer": "gold",
        "depends_on": ["silver_products"],
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_product",
        "target_table": "gbmart.gold.dim_product",
    },
    {
        "task_key": "gold_dim_address",
        "layer": "gold",
        "depends_on": ["silver_address"],
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_address",
        "target_table": "gbmart.gold.dim_address",
    },
    {
        "task_key": "gold_dim_payment_method",
        "layer": "gold",
        "depends_on": ["silver_payment_methods"],
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_payment_method",
        "target_table": "gbmart.gold.dim_payment_method",
    },
    {
        "task_key": "gold_dim_orders",
        "layer": "gold",
        "depends_on": ["silver_orders"],
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_orders",
        "target_table": "gbmart.gold.dim_orders",
    },
    {
        "task_key": "gold_dim_date",
        "layer": "gold",
        "depends_on": ["silver_orders"],   # date spine is sized from silver.orders' real date range
        "notebook_path": "/Repos/globalmart-prod/gold/build_dim_date",
        "target_table": "gbmart.gold.dim_date",
    },

    # -- Gold: fact_sales -- the real fan-in. Waits on 5 Silver tables AND
    # gold_dim_date (a GOLD task, not Silver). This edge deliberately skips
    # straight from Gold to Gold -- real DAG edges follow data dependency
    # only, they don't have to respect neat "layer by layer" boundaries.
    {
        "task_key": "gold_fact_sales",
        "layer": "gold",
        "depends_on": [
            "silver_order_items", "silver_orders", "silver_products",
            "silver_address", "silver_payments", "gold_dim_date",
        ],
        "notebook_path": "/Repos/globalmart-prod/gold/build_fact_sales",
        "target_table": "gbmart.gold.fact_sales",
    },
]

print(f"Total tasks in GlobalMart's real job: {len(globalmart_tasks)}")
for layer in ("bronze", "silver", "gold"):
    layer_tasks = [t["task_key"] for t in globalmart_tasks if t["layer"] == layer]
    print(f"  {layer:7s}: {len(layer_tasks)} tasks -> {layer_tasks}")

In [ ]:
# --- Display the graph as depends_on edges (no Spark -- print only) --------
print(f"{'task_key':26s} {'layer':7s} depends_on")
print("-" * 90)
for t in globalmart_tasks:
    deps = ", ".join(t["depends_on"]) if t["depends_on"] else "(none -- ready immediately)"
    print(f"{t['task_key']:26s} {t['layer']:7s} {deps}")

---
## Section 5: Topological Sort — Computing a Valid Run Order, For Real

This is the actual algorithm every real orchestrator runs before executing anything: given a DAG, produce an order where every task appears only after all of its dependencies. When more than one valid order exists — which is normal, that's exactly what fan-out means — a useful implementation also reports which tasks are free to run **at the same time**.

The function below is **Kahn's algorithm**, written in plain Python: repeatedly collect every task whose dependencies are all already satisfied, treat that whole group as one parallel "wave," then remove them and repeat. If tasks remain but none of them ever becomes free, the graph has a cycle — Section 6 triggers this on purpose and shows the resulting error.

In [ ]:
def topological_sort(tasks):
    """
    Pure-Python topological sort using Kahn's algorithm.

    tasks : list of dicts, each with at least
            "task_key"   (str)        -- unique id for this task
            "depends_on" (list[str])  -- upstream task_keys

    Returns (flat_order, waves):
        flat_order -- a single valid end-to-end run order (list of task_key)
        waves      -- list of lists of task_key; everything inside waves[i]
                      can run in parallel, and only after every task in
                      waves[0..i-1] has finished. len(waves[i]) > 1 is
                      fan-out, made concrete.

    Raises ValueError if the graph contains a cycle, or references an
    unknown task_key -- a real scheduler must detect this instead of
    waiting forever for a task that can never become ready.
    """
    task_by_key = {t["task_key"]: t for t in tasks}
    if len(task_by_key) != len(tasks):
        raise ValueError("Duplicate task_key found -- every task_key must be unique.")

    in_degree = {key: 0 for key in task_by_key}      # unfinished upstream deps, per task
    children = {key: [] for key in task_by_key}      # reverse edges: who unblocks when key finishes

    for t in tasks:
        for upstream in t["depends_on"]:
            if upstream not in task_by_key:
                raise ValueError(f"'{t['task_key']}' depends_on unknown task_key '{upstream}'")
            in_degree[t["task_key"]] += 1
            children[upstream].append(t["task_key"])

    ready = sorted(key for key, deg in in_degree.items() if deg == 0)
    remaining = dict(in_degree)

    flat_order = []
    waves = []
    while ready:
        waves.append(ready)
        flat_order.extend(ready)
        next_ready = []
        for key in ready:
            for child in children[key]:
                remaining[child] -= 1
                if remaining[child] == 0:
                    next_ready.append(child)
        ready = sorted(next_ready)

    if len(flat_order) != len(tasks):
        stuck = sorted(set(task_by_key) - set(flat_order))
        raise ValueError(
            "Cycle detected -- these tasks can never become ready because "
            f"they depend on each other, directly or indirectly: {stuck}"
        )

    return flat_order, waves

print("topological_sort() defined -- pure Python, no imports needed.")

In [ ]:
flat_order, waves = topological_sort(globalmart_tasks)

print("A valid end-to-end run order for GlobalMart's real pipeline:\n")
for i, key in enumerate(flat_order, start=1):
    print(f"  {i:2d}. {key}")

print(f"\n{len(waves)} parallel waves -- fan-out/fan-in, computed automatically:\n")
for i, wave in enumerate(waves):
    print(f"  Wave {i}: {wave}")
    print(f"           ({len(wave)} task{'s' if len(wave) != 1 else ''} can run in parallel here)")

assert flat_order[-1] == "gold_fact_sales", "fact_sales should always land last -- it depends, directly or indirectly, on everything else."
print("\nSanity check passed: gold_fact_sales is scheduled last, as expected.")

---
## Section 6: Why DAGs, Not Arbitrary Graphs — Proving It With a Real Error

Take three of GlobalMart's real task names and wire them into a **cycle**, on purpose: `bronze_orders → silver_orders → gold_dim_orders → bronze_orders`. This is deliberately invalid. Watch `topological_sort` refuse to schedule it, with a clear, caught error — not an infinite wait, and not a crash that would break this notebook's "Run All."

In [ ]:
# A deliberately broken graph: gold_dim_orders is wired back to depend on
# bronze_orders, but bronze_orders itself is wired to depend on gold_dim_orders --
# a 3-node cycle. This must never be schedulable -- that's the entire point.
broken_tasks = [
    {"task_key": "bronze_orders",   "depends_on": ["gold_dim_orders"]},   # <- the illegal edge
    {"task_key": "silver_orders",   "depends_on": ["bronze_orders"]},
    {"task_key": "gold_dim_orders", "depends_on": ["silver_orders"]},
]

try:
    topological_sort(broken_tasks)
    print("This line should never print -- a cycle should always raise ValueError.")
except ValueError as e:
    print("Caught the expected error -- exactly what a real scheduler should do")
    print("instead of hanging forever waiting for a task that can never become ready:\n")
    print(f"  ValueError: {e}")

---
## Section 7: What This Would Look Like As a Real Databricks Job (Display Only)

The cell below reshapes GlobalMart's task list into the nested shape the real Databricks Jobs REST API expects for a job-creation request — `depends_on` becomes a list of `{"task_key": ...}` objects, and each task carries either a `notebook_task` or a `pipeline_task` block.

**This is printed as text. Nothing is sent anywhere.** The function below builds and returns a plain Python `dict`; the only thing that happens to it afterward is `json.dumps(...)` inside a `print()` statement. There is no import of any Databricks SDK, no `requests` call, no workspace client, and no job-creation or run-triggering call of any kind, in this cell or anywhere else in this notebook. Building this JSON is free; *submitting* it would create a real, running, billable Databricks Job — which is precisely the line this notebook does not cross.

In [ ]:
import json

# --- Reshape into the real Databricks Jobs API task schema -- DISPLAY ONLY -
# This function only builds and returns a plain Python dict. It performs no
# I/O of any kind: no HTTP request, no SDK client, no authentication, no
# workspace connection. The result is only ever handed to json.dumps() and
# printed below -- never submitted anywhere, by this cell or any other cell
# in this notebook.
def to_databricks_job_json(tasks, job_name):
    api_tasks = []
    for t in tasks:
        api_task = {
            "task_key": t["task_key"],
            "depends_on": [{"task_key": d} for d in t["depends_on"]],
        }
        if "pipeline_id" in t:
            api_task["pipeline_task"] = {"pipeline_id": t["pipeline_id"]}
        else:
            api_task["notebook_task"] = {"notebook_path": t["notebook_path"], "source": "WORKSPACE"}
        api_tasks.append(api_task)

    return {
        "name": job_name,
        "tasks": api_tasks,
        "job_clusters": [{
            "job_cluster_key": "globalmart_shared_cluster",
            "new_cluster": {"spark_version": "15.4.x-scala2.12", "num_workers": 2},
        }],
        "schedule": {"quartz_cron_expression": "0 0 6 * * ?", "timezone_id": "Asia/Kolkata"},
    }

job_definition = to_databricks_job_json(globalmart_tasks, job_name="globalmart_daily_medallion_pipeline")

print(json.dumps(job_definition, indent=2))

---
## Section 8: How You'd Actually Submit This (Never From Inside This Notebook)

Turning the JSON above into a real, running Databricks Job is a deliberate action a person takes **outside this notebook entirely** — for example, by pasting it into the Workflows UI's "Create Job" JSON view, or via the Databricks CLI:

```bash
# Illustrative only -- shown so you recognize this exact step when Day 11's
# Hands-On walks you through it for real, deliberately, with the instructor
# watching. Do not run this against a shared workspace outside that session.
databricks jobs create --json @globalmart_job.json
```

**Why this notebook never does this itself, not even once:** creating a job costs nothing by itself, but a job is *meant* to run — and one left attached to a `schedule` block like Section 4/7's will trigger real, unattended cluster starts on its own timetable, every day, until a person deletes it. That's real, ongoing compute spend with nobody watching it happen. A conceptual ILT session that every future cohort re-runs via "Run All" is exactly the wrong place for an action with that consequence. Day 11 creates a real job exactly once, deliberately, paused or manually-triggered rather than scheduled — never as a side effect of opening a notebook.

---
## Key Takeaways

| Topic | Key Takeaway |
|---|---|
| **The pain point** | GlobalMart's pipeline is already 12+ real hands-on notebooks across 7 Bronze, 7 Silver, and 7 Gold tables, each with strict ordering rules a human currently has to remember |
| **DAG** | Nodes (tasks) + directed edges (dependencies), and it must be acyclic — or no valid first task exists |
| **Fan-out** | Free parallelism — GlobalMart's 7 Bronze tasks, then its 7 Silver tasks, are each mutually independent |
| **Fan-in** | A synchronization point — `gold_fact_sales` alone waits on 6 upstream tasks at once |
| **A Databricks Job** | Just a JSON/dict of tasks — `task_key`, `depends_on`, and a `notebook_task` or `pipeline_task` — exactly what Section 4/7 built and printed |
| **Topological sort** | What makes a DAG schedulable — computes a valid order (and valid parallel waves) automatically, and fails loudly on a cycle instead of hanging |
| **Only 2 real pathways** | API/GraphDB patterns from Day 3 don't get a `task_key` here, on purpose — they never write to a real `gbmart` table |
| **Nothing here was billable** | Every cell was plain Python, executed entirely in memory — no cluster, job, pipeline, or warehouse was created, started, or scheduled |

## What's Next

**Day 11 — Hands-On: Building the Real GlobalMart Workflow.** You'll take the exact task graph designed today and create it for real, in the Databricks Workflows UI (or CLI) — wiring up every `task_key` and `depends_on` edge, on a paused or manually-triggered schedule so nothing runs unattended or unexpectedly. Bring today's `globalmart_tasks` list; you'll be translating it almost line-by-line into real clicks.

### Discussion Questions

1. Day 5's real Silver build also checks `order_items` and `payments` for referential integrity against `silver.orders`/`silver.products`, which must already exist when that check runs. If you added those as real `depends_on` edges — instead of each table depending only on its own Bronze parent — which wave would `silver_order_items` and `silver_payments` move into? Would `gold_fact_sales` still be the last task to run?
2. Section 2 mentioned a coarser alternative design: every Gold task depends on *all seven* Silver tasks finishing, instead of just its own one. Rebuild `globalmart_tasks` that way in your head (or in a scratch cell) — what happens to the number of waves, and to how long `dim_payment_method` has to wait before it can start?
3. `bronze_orders` and `bronze_order_items` were modeled with a `pipeline_id`, not a `notebook_path`. Why does that distinction matter to someone actually configuring this job in the Databricks Workflows UI?
4. Why does `topological_sort` need to raise an error the moment it detects a cycle, rather than simply skipping the tasks it can't resolve and scheduling everything else?